<a href="https://colab.research.google.com/github/Engr-Muhammad-Anees/Dubbing-Podcast-ML/blob/main/db_podcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install moviepy assemblyai elevenlabs pydub

In [ ]:
import time
import os
import io
import assemblyai as aai
from moviepy.editor import VideoFileClip, AudioFileClip
from elevenlabs import ElevenLabs
from pydub import AudioSegment

In [ ]:
ASSEMBLYAI_API_KEY = "10f21c29a6784597b2707e8566bf491b"
ELEVENLABS_API_KEY = "sk_ceb7e77743b0030dc2c75bfb4793b1f0eeb02e79949589dc"

In [ ]:
video_path = "/content/WhatsApp Video 2025-11-10 at 2.11.00 PM.mp4" # input video

In [ ]:
input_video_clip = VideoFileClip(video_path)
audio_path = "extracted_audio.wav"
input_video_clip.audio.write_audiofile(audio_path, verbose=False, logger=None)
print(f" Video duration: {input_video_clip.duration:.2f}s")

 Video duration: 22.22s


In [ ]:
audio_path = "/content/extracted_audio.wav" # extract audio path

In [ ]:
speaker_voices = {
    "A": "TxGEqnHWrfWFTfGW9XjX",             #for AI voice select
    "SPEAKER_0": "TxGEqnHWrfWFTfGW9XjX"
}

In [ ]:
aai.settings.api_key = ASSEMBLYAI_API_KEY
transcriber = aai.Transcriber()
config = aai.TranscriptionConfig(speaker_labels=True, speakers_expected=1)  #use assembly aai for transcript
transcript = transcriber.transcribe(audio_path, config)

segments = transcript.utterances
print(f"Total segment detect: {len(segments)}")

Total segment detect: 9


In [ ]:
segments # for checking segment

In [ ]:
total_duration_ms = int(video.duration * 1000)
final_dub_audio = AudioSegment.silent(duration=total_duration_ms) # Create an empty silent audio
voice_id = "pNInz6obpgDQGcFmaJgB"
for seg in segments: # Loop through each segment
    text = seg.text.strip()
    start_ms = int(seg.start)
    end_ms = int(seg.end)
    duration_ms = max(200, end_ms - start_ms)  # minimum segment duration

    print(f"\n Generating audio ({start_ms}-{end_ms} ms | {duration_ms/1000:.2f}s): '{text}'")

    seg_audio = None
    attempt = 0
    while attempt < 2 and seg_audio is None:
        try:
            audio_stream = client.text_to_speech.convert( # Request audio from ElevenLabs API
                voice_id=voice_id,
                model_id="eleven_multilingual_v2",
                text=text
            )

            audio_bytes = b"".join(audio_stream)  # Combine streamed chunks into a single byte sequence
            if not audio_bytes:
                raise ValueError("Empty stream received from ElevenLabs")
            seg_audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")  # Convert byte data to an AudioSegment object

        except Exception as e:
            attempt += 1
            print(f" Error (attempt {attempt}): {e}")
            time.sleep(2)
    if seg_audio is None:   # If TTS generation fails, insert silence for this segment
        print(f" ElevenLabs failed for '{text}', inserting {duration_ms}ms silence.")
        seg_audio = AudioSegment.silent(duration=duration_ms)

    # generated audio to the final track at the correct position
    final_dub_audio = final_dub_audio.overlay(seg_audio, position=start_ms)
dubbed_ms = len(final_dub_audio)           # Ensure the final audio matches the exact video duration
video_ms = int(video.duration * 1000)

if dubbed_ms < video_ms:
    final_dub_audio += AudioSegment.silent(duration=video_ms - dubbed_ms)   # Pad with silence if audio is shorter than video
elif dubbed_ms > video_ms:
    final_dub_audio = final_dub_audio[:video_ms]    # Trim excess audio if it’s longer than video
if not path.endswith('.wav'):                       # Ensure output file is in WAV format
    dubb_audio = os.path.splitext(path)[0] + '.wav'

final_dub_audio.export(path, format="wav")           # Export the final dubbed audio file
print(f"\n Dubbed audio saved at: {path}")


In [ ]:
client = ElevenLabs(api_key=ELEVENLABS_API_KEY) # for client using api

In [ ]:
path = "/content/drive/MyDrive/dping_podcast/final_audio4.wav" #path of dubbed_audio

In [ ]:
dubbed_audio = "/content/drive/MyDrive/dping_podcast/final_audio4.wav"

In [ ]:
video = VideoFileClip(video_path)
audio = AudioFileClip(dubbed_audio)
final_video = video.set_audio(audio)
output_path = "/content/merged_video.mp4"
final_video.write_videofile(output_path, codec="libx264", audio_codec="aac")

print(" Merging complete:", output_path)